# 🐦‍⬛ RAVEN-2 — Full Training Run
**50GB corpus | 100M params | Drive checkpoints | ~6hr on T4**

### Order of operations
1. Mount Drive + Upload zip  
2. Install deps  
3. Download 40GB web corpus (fast `wget` shards — no HF API hang)  
4. Generate 1M personality pairs → repeat to 10GB  
5. Merge corpus → train tokenizer → pack binary  
6. Pretrain 13K steps (~5hrs)  
7. Instruct tune → Posttrain  
8. Download final `.pt` weight

## ⚙️ Step 0 — Check GPU & Mount Drive

In [ ]:
import subprocess, os, shutil, time
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
print("GPU:", result.stdout.strip())

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create Drive checkpoint dir
DRIVE_CKPT = "/content/drive/MyDrive/Raven2/checkpoints"
DRIVE_DATA = "/content/drive/MyDrive/Raven2/data"
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)
print("✅ Drive mounted. Checkpoint dir:", DRIVE_CKPT)

## 📦 Step 1 — Upload Raven-2 zip & Install deps

In [ ]:
# Upload your zip manually using the file panel on the left,
# OR run this cell to get a file picker:
from google.colab import files
uploaded = files.upload()  # pick your Raven-2 zip

import zipfile, os
for fname in uploaded:
    print(f"Extracting {fname}...")
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/')
    print("Done.")

# Find the extracted root (handles nested folders)
for root, dirs, fs in os.walk('/content'):
    if 'config.py' in fs and 'model.py' in fs:
        PROJECT_ROOT = root
        break

os.chdir(PROJECT_ROOT)
print("✅ Project root:", PROJECT_ROOT)
os.makedirs("data", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

In [ ]:
# Install dependencies
import subprocess
subprocess.run(["pip", "install", "-q",
    "torch>=2.1.0", "tokenizers>=0.15.0", "numpy>=1.24.0",
    "tqdm>=4.65.0", "datasets>=2.14.0", "huggingface_hub>=0.20.0",
    "requests>=2.28.0"], check=True)
print("✅ Dependencies installed.")

## 🚀 Step 2 — Download 40GB Web Corpus (fast `wget` direct shards)

**Why this is different from the original script:**  
The original `get_fast_dataset()` calls `HfApi().list_repo_files()` which hangs on free Colab due to rate limits and slow API resolution.  
This cell uses **direct `wget`** on known parquet shard URLs — hits full Colab network speed (~100–200 MB/s) with no API overhead.

In [ ]:
import os, subprocess, threading, time

# ── CONFIG ──
DATA_DIR = "data/raw_parquet"
os.makedirs(DATA_DIR, exist_ok=True)

# Direct parquet shard URLs — no HF Hub API needed, pure wget
# FineWeb-Edu sample-10BT: 50% of corpus
FINEWEB_BASE = "https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/main/sample/10BT"
FINEWEB_SHARDS = [f"train-{str(i).zfill(5)}-of-00099.parquet" for i in range(50)]  # ~20GB text

# Wikipedia EN 2023: 25% of corpus
WIKI_BASE = "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/20231101.en"
WIKI_SHARDS = [f"train-{str(i).zfill(5)}-of-00041.parquet" for i in range(20)]  # ~8GB text

# Cosmopedia v2: 15% of corpus
COSMO_BASE = "https://huggingface.co/datasets/HuggingFaceTB/cosmopedia-v2/resolve/main/data"
COSMO_SHARDS = [f"train-{str(i).zfill(5)}-of-00212.parquet" for i in range(18)]  # ~6GB text

# Gutenberg: 10% of corpus
GUTENBERG_BASE = "https://huggingface.co/datasets/sedthh/gutenberg_english/resolve/main/data"
GUTENBERG_SHARDS = [f"train-{str(i).zfill(5)}-of-00025.parquet" for i in range(10)]  # ~4GB text

ALL_DOWNLOADS = (
    [(FINEWEB_BASE, s, "fineweb") for s in FINEWEB_SHARDS] +
    [(WIKI_BASE, s, "wiki") for s in WIKI_SHARDS] +
    [(COSMO_BASE, s, "cosmo") for s in COSMO_SHARDS] +
    [(GUTENBERG_BASE, s, "gutenberg") for s in GUTENBERG_SHARDS]
)

print(f"Total shards to download: {len(ALL_DOWNLOADS)}")
print(f"Estimated corpus size: ~40GB text after extraction")

def download_shard(base_url, shard_name, prefix, idx, total, results):
    dest_path = os.path.join(DATA_DIR, f"{prefix}_{shard_name}")
    if os.path.exists(dest_path):
        results[idx] = ("skip", dest_path)
        return
    url = f"{base_url}/{shard_name}"
    ret = subprocess.run(
        ["wget", "-q", "--retry-connrefused", "--tries=5",
         "--waitretry=3", "-O", dest_path, url],
        capture_output=True
    )
    if ret.returncode == 0 and os.path.getsize(dest_path) > 10000:
        results[idx] = ("ok", dest_path)
    else:
        if os.path.exists(dest_path):
            os.remove(dest_path)
        results[idx] = ("fail", url)

# Download in parallel batches of 8 threads
BATCH = 8
results = [None] * len(ALL_DOWNLOADS)
t_start = time.time()

for batch_start in range(0, len(ALL_DOWNLOADS), BATCH):
    batch = ALL_DOWNLOADS[batch_start:batch_start+BATCH]
    threads = []
    for i, (base, shard, prefix) in enumerate(batch):
        idx = batch_start + i
        t = threading.Thread(target=download_shard, args=(base, shard, prefix, idx, len(ALL_DOWNLOADS), results))
        threads.append(t)
        t.start()
    for t in threads:
        t.join()

    done = batch_start + len(batch)
    elapsed = time.time() - t_start
    # Disk usage
    total_bytes = sum(os.path.getsize(os.path.join(DATA_DIR, f))
                      for f in os.listdir(DATA_DIR)
                      if os.path.isfile(os.path.join(DATA_DIR, f)))
    speed = total_bytes / elapsed / 1e6 if elapsed > 0 else 0
    print(f"  [{done}/{len(ALL_DOWNLOADS)}] {total_bytes/1e9:.2f} GB on disk | {speed:.1f} MB/s | {elapsed/60:.1f} min")

failed = [r[1] for r in results if r and r[0] == "fail"]
skipped = sum(1 for r in results if r and r[0] == "skip")
print(f"\n✅ Download complete. Failed: {len(failed)} | Skipped (already existed): {skipped}")
if failed:
    print("Failed URLs:", failed[:5])

## 📝 Step 3 — Extract parquet shards → `data/corpus.txt`

In [ ]:
import os, re, unicodedata, hashlib
from tqdm import tqdm

DATA_DIR    = "data/raw_parquet"
CORPUS_FILE = "data/corpus.txt"

def clean(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

def ok(text):
    if len(text) < 200: return False
    words = text.split()
    if len(words) < 50: return False
    alpha = sum(c.isalpha() or c.isspace() for c in text)
    if alpha / len(text) < 0.70: return False
    digits = sum(c.isdigit() for c in text)
    if digits / len(text) > 0.10: return False
    return True

def prefix_hash(text):
    return hashlib.md5(text[:1000].strip().encode()).hexdigest()

parquet_files = sorted([
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith(".parquet")
])

print(f"Found {len(parquet_files)} parquet shards to process.")

try:
    import pyarrow.parquet as pq
    USE_PYARROW = True
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "pyarrow"], check=True)
    import pyarrow.parquet as pq
    USE_PYARROW = True

dedup = set()
total_written = 0
WRITE_BUFFER_SIZE = 8 * 1024 * 1024  # 8MB

# Resume: rebuild dedup from existing corpus
if os.path.exists(CORPUS_FILE):
    existing_size = os.path.getsize(CORPUS_FILE)
    print(f"Resuming — existing corpus: {existing_size/1e9:.2f} GB. Rebuilding dedup index...")
    chunk_size = 10 * 1024 * 1024
    buf = ""
    with open(CORPUS_FILE, "r", encoding="utf-8", errors="ignore") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk: break
            buf += chunk
            docs = buf.split("<|eos|>")
            buf = docs.pop()
            for doc in docs:
                d = doc.strip()
                if d: dedup.add(prefix_hash(d))
    total_written = existing_size
    print(f"  Dedup index: {len(dedup):,} docs")

with open(CORPUS_FILE, "a", encoding="utf-8") as out:
    write_buf = []
    buf_chars = 0

    for pf in tqdm(parquet_files, desc="Extracting shards"):
        try:
            table = pq.read_table(pf)
        except Exception as e:
            print(f"  Skipping corrupt shard {pf}: {e}")
            continue

        # Find text column (varies by dataset)
        col = None
        for candidate in ["text", "content", "TEXT", "body"]:
            if candidate in table.column_names:
                col = candidate
                break
        if col is None:
            col = table.column_names[0]

        for row in table.column(col).to_pylist():
            if not isinstance(row, str): continue
            text = clean(row)
            if not ok(text): continue
            h = prefix_hash(text)
            if h in dedup: continue
            dedup.add(h)
            entry = f"{text}<|eos|>\n\n"
            write_buf.append(entry)
            buf_chars += len(entry)
            total_written += len(entry.encode())
            if buf_chars >= WRITE_BUFFER_SIZE:
                out.write("".join(write_buf))
                out.flush()
                write_buf = []
                buf_chars = 0

    if write_buf:
        out.write("".join(write_buf))

corpus_gb = os.path.getsize(CORPUS_FILE) / 1e9
print(f"\n✅ corpus.txt ready: {corpus_gb:.2f} GB | {len(dedup):,} unique docs")

## 🎭 Step 4 — Generate 1M personality pairs → expand to 10GB

In [ ]:
import os, subprocess, sys

# Generate 1M pairs using existing script
print("Generating 1,000,000 personality pairs (~400MB)...")
result = subprocess.run(
    [sys.executable, "data/build_personality_corpus.py", "--count", "1000000"],
    capture_output=False
)
print("Exit code:", result.returncode)

personality_file = "data/personality.txt"
if not os.path.exists(personality_file):
    print("❌ personality.txt not found. Check build_personality_corpus.py output.")
else:
    size_mb = os.path.getsize(personality_file) / 1e6
    print(f"  personality.txt: {size_mb:.1f} MB")

In [ ]:
import os
# Now expand personality.txt to ~10GB by repeating it into personality_expanded.txt
# This gives the pretrain loop 10GB of personality-flavored text

PERSONALITY_SRC = "data/personality.txt"
PERSONALITY_EXP = "data/personality_expanded.txt"
TARGET_BYTES    = 10 * 1024 * 1024 * 1024  # 10GB

src_size = os.path.getsize(PERSONALITY_SRC)
repeats  = (TARGET_BYTES // src_size) + 1
print(f"Source: {src_size/1e6:.1f} MB — repeating {repeats}x to reach 10GB...")

written = 0
WRITE_CHUNK = 64 * 1024 * 1024  # 64MB read chunks
with open(PERSONALITY_EXP, "wb") as out:
    for rep in range(repeats):
        if written >= TARGET_BYTES:
            break
        with open(PERSONALITY_SRC, "rb") as src:
            while True:
                chunk = src.read(WRITE_CHUNK)
                if not chunk:
                    break
                remaining = TARGET_BYTES - written
                if len(chunk) > remaining:
                    chunk = chunk[:remaining]
                out.write(chunk)
                written += len(chunk)
                if written >= TARGET_BYTES:
                    break
        if rep % 5 == 0:
            print(f"  {written/1e9:.2f} GB written...")

final_size = os.path.getsize(PERSONALITY_EXP)
print(f"\n✅ personality_expanded.txt: {final_size/1e9:.2f} GB")

## 🔀 Step 5 — Merge web corpus + personality into final pretrain corpus

In [ ]:
import os, subprocess, sys

CORPUS_FILE  = "data/corpus.txt"
PERSONALITY  = "data/personality_expanded.txt"
MERGED_FILE  = "data/pretrain_corpus.txt"

corpus_gb = os.path.getsize(CORPUS_FILE) / 1e9
pers_gb   = os.path.getsize(PERSONALITY) / 1e9
print(f"Web corpus:   {corpus_gb:.2f} GB")
print(f"Personality:  {pers_gb:.2f} GB")
print(f"Merging into: {MERGED_FILE}")

# Interleave them: write corpus first, then personality
# (data_prep.py streams the file, so order doesn't matter)
CHUNK = 64 * 1024 * 1024
total_written = 0
with open(MERGED_FILE, "wb") as out:
    for src_path in [CORPUS_FILE, PERSONALITY]:
        src_size = os.path.getsize(src_path)
        with open(src_path, "rb") as f:
            while True:
                chunk = f.read(CHUNK)
                if not chunk: break
                out.write(chunk)
                total_written += len(chunk)
        print(f"  ✅ {src_path} merged ({src_size/1e9:.2f} GB)")

final_gb = os.path.getsize(MERGED_FILE) / 1e9
print(f"\n✅ pretrain_corpus.txt: {final_gb:.2f} GB total")

## 🔤 Step 6 — Train BPE tokenizer

In [ ]:
import os, sys, subprocess

TOKENIZER_PATH = "checkpoints/raven2_tokenizer.json"

# Check if Drive already has a trained tokenizer (resume case)
DRIVE_TOK = "/content/drive/MyDrive/Raven2/checkpoints/raven2_tokenizer.json"
if not os.path.exists(TOKENIZER_PATH) and os.path.exists(DRIVE_TOK):
    import shutil
    shutil.copy(DRIVE_TOK, TOKENIZER_PATH)
    print("✅ Tokenizer restored from Drive.")

if os.path.exists(TOKENIZER_PATH):
    print("✅ Tokenizer already exists, skipping training.")
else:
    print("Training tokenizer on pretrain_corpus.txt (sample)...")
    # Train on first 1GB of the merged corpus for speed
    SAMPLE = "data/tokenizer_sample.txt"
    SAMPLE_SIZE = 1 * 1024 * 1024 * 1024
    with open("data/pretrain_corpus.txt", "rb") as f, open(SAMPLE, "wb") as out:
        written = 0
        CHUNK = 64 * 1024 * 1024
        while written < SAMPLE_SIZE:
            chunk = f.read(CHUNK)
            if not chunk: break
            out.write(chunk)
            written += len(chunk)
    print(f"  Sample: {os.path.getsize(SAMPLE)/1e6:.0f} MB")

    # Patch tokenizer_train.py to use the sample file
    import tokenizer_train
    tokenizer_train.train_tokenizer(SAMPLE, TOKENIZER_PATH)
    print("✅ Tokenizer trained.")

    # Backup to Drive
    import shutil
    shutil.copy(TOKENIZER_PATH, DRIVE_TOK)
    print("☁️  Tokenizer backed up to Drive.")

## 🗜️ Step 7 — Pack corpus into binary memmap files

In [ ]:
import os, sys, shutil, subprocess

TRAIN_BIN = "checkpoints/train.bin"
VAL_BIN   = "checkpoints/val.bin"
DRIVE_CKPT = "/content/drive/MyDrive/Raven2/checkpoints"

# Restore from Drive if available
for fname in ["train.bin", "val.bin"]:
    drive_path = os.path.join(DRIVE_CKPT, fname)
    local_path = f"checkpoints/{fname}"
    if not os.path.exists(local_path) and os.path.exists(drive_path):
        print(f"Restoring {fname} from Drive...")
        shutil.copy(drive_path, local_path)

if os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN):
    train_gb = os.path.getsize(TRAIN_BIN) / 1e9
    val_gb   = os.path.getsize(VAL_BIN) / 1e9
    print(f"✅ Binary files already exist: train.bin {train_gb:.2f}GB | val.bin {val_gb:.2f}GB")
    print("Skipping data prep.")
else:
    print("Packing pretrain_corpus.txt into binary files...")
    # Patch data_prep to use our merged corpus
    result = subprocess.run(
        [sys.executable, "data/data_prep.py", "--mode", "pretrain",
         "--inputs", "data/pretrain_corpus.txt"],
        capture_output=False
    )
    print("Exit code:", result.returncode)
    # Backup to Drive
    for fname in ["train.bin", "val.bin"]:
        src = f"checkpoints/{fname}"
        if os.path.exists(src):
            shutil.copy(src, os.path.join(DRIVE_CKPT, fname))
            print(f"☁️  {fname} backed up to Drive.")

## 🗜️ Step 7b — Pack instruct data

In [ ]:
import os, sys, shutil, subprocess

INSTRUCT_BIN = "checkpoints/instruct.bin"
DRIVE_CKPT = "/content/drive/MyDrive/Raven2/checkpoints"

# Restore from Drive
for fname in ["instruct.bin", "instruct_val.bin"]:
    drive_path = os.path.join(DRIVE_CKPT, fname)
    local_path = f"checkpoints/{fname}"
    if not os.path.exists(local_path) and os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
        print(f"Restored {fname} from Drive.")

if os.path.exists(INSTRUCT_BIN):
    print("✅ instruct.bin already exists, skipping.")
else:
    print("Building instruct dataset...")
    subprocess.run([sys.executable, "data/build_instruct_data.py"], check=True)
    subprocess.run([sys.executable, "data/data_prep.py", "--mode", "instruct"], check=True)
    for fname in ["instruct.bin", "instruct_val.bin"]:
        src = f"checkpoints/{fname}"
        if os.path.exists(src):
            shutil.copy(src, os.path.join(DRIVE_CKPT, fname))
    print("✅ Instruct binary ready.")

## 🧠 Step 8 — Pretrain (~5 hours)

**Config overrides:**
- `max_iters = 13000` (fits 6hr budget — full 50K would take ~18hrs)
- `eval_interval = 250` (more frequent Drive backups)
- All other hyperparams from `config.py` unchanged

In [ ]:
import os, sys, shutil, torch
import config

DRIVE_CKPT = "/content/drive/MyDrive/Raven2/checkpoints"
LATEST_CKPT = "checkpoints/raven2_pretrain_latest.pt"

# Restore latest checkpoint from Drive if this is a resumed session
drive_latest = os.path.join(DRIVE_CKPT, "raven2_pretrain_latest.pt")
if not os.path.exists(LATEST_CKPT) and os.path.exists(drive_latest):
    print("Restoring latest pretrain checkpoint from Drive...")
    shutil.copy(drive_latest, LATEST_CKPT)
    print("✅ Checkpoint restored. Will resume from last step.")

# Override config for 6hr run
config.max_iters     = 13000
config.eval_interval = 250
config.eval_iters    = 50
config.drive_backup_dir = DRIVE_CKPT

print(f"Device: {config.device}")
print(f"AMP:    {config.use_amp}")
print(f"Steps:  {config.max_iters:,}")
print(f"Batch:  {config.batch_size} x {config.gradient_accumulation_steps} accum = {config.batch_size * config.gradient_accumulation_steps} effective")
print(f"Tokens/step: {config.batch_size * config.gradient_accumulation_steps * config.block_size:,}")
print("\n🚀 Starting pretrain...")

In [ ]:
# Run pretrain — auto-resumes if checkpoint exists
import train
train.main()

## 🎯 Step 9 — Instruction Fine-tuning (~20 min)

In [ ]:
import os, shutil, config

DRIVE_CKPT = "/content/drive/MyDrive/Raven2/checkpoints"

# Restore pretrain checkpoint if needed (new Colab session)
if not os.path.exists(config.pretrain_ckpt):
    drive_ckpt = os.path.join(DRIVE_CKPT, "raven2_pretrain_best.pt")
    if os.path.exists(drive_ckpt):
        shutil.copy(drive_ckpt, config.pretrain_ckpt)
        print("✅ Pretrain checkpoint restored from Drive.")
    else:
        print("❌ No pretrain checkpoint found! Run Step 8 first.")

config.instruct_max_iters = 5000
config.drive_backup_dir   = DRIVE_CKPT

import instruct_tune
instruct_tune.main()

## 🐦 Step 10 — Personality Post-training (~8 min)

In [ ]:
import os, sys, subprocess, shutil, config

DRIVE_CKPT = "/content/drive/MyDrive/Raven2/checkpoints"
POSTTRAIN_BIN = "checkpoints/posttrain.bin"

# Build posttrain binary from personality.txt if needed
if not os.path.exists(POSTTRAIN_BIN):
    print("Building posttrain binary from personality.txt...")
    subprocess.run([
        sys.executable, "data/data_prep.py",
        "--mode", "instruct",
        "--inputs", "data/personality.txt"
    ], check=True)
    # data_prep instruct mode writes instruct.bin — move to posttrain.bin
    if os.path.exists("checkpoints/instruct.bin"):
        shutil.copy("checkpoints/instruct.bin", POSTTRAIN_BIN)
    print("✅ posttrain.bin ready.")

# Restore instruct checkpoint if needed
if not os.path.exists(config.instruct_ckpt):
    drive_ckpt = os.path.join(DRIVE_CKPT, "raven2_instruct_best.pt")
    if os.path.exists(drive_ckpt):
        shutil.copy(drive_ckpt, config.instruct_ckpt)
        print("✅ Instruct checkpoint restored from Drive.")

config.posttrain_max_iters = 2000
config.drive_backup_dir    = DRIVE_CKPT

import posttrain
posttrain.main()

## 📊 Step 11 — Quick Eval

In [ ]:
import os, sys, shutil, config

DRIVE_CKPT = "/content/drive/MyDrive/Raven2/checkpoints"

# Restore final checkpoint if needed
if not os.path.exists(config.posttrain_ckpt):
    drive_ckpt = os.path.join(DRIVE_CKPT, "raven2_posttrain_best.pt")
    if os.path.exists(drive_ckpt):
        shutil.copy(drive_ckpt, config.posttrain_ckpt)

import subprocess
subprocess.run([sys.executable, "eval/run_full_eval.py"])

## 💾 Step 12 — Download final checkpoint

In [ ]:
import os, shutil, config
from google.colab import files

DRIVE_CKPT = "/content/drive/MyDrive/Raven2/checkpoints"

# Pick best available checkpoint: posttrain > instruct > pretrain
for ckpt_path in [config.posttrain_ckpt, config.instruct_ckpt, config.pretrain_ckpt]:
    if os.path.exists(ckpt_path):
        # Also make sure Drive copy is current
        shutil.copy(ckpt_path, os.path.join(DRIVE_CKPT, os.path.basename(ckpt_path)))
        size_mb = os.path.getsize(ckpt_path) / 1e6
        print(f"Downloading {ckpt_path} ({size_mb:.0f} MB)...")
        files.download(ckpt_path)
        print("✅ Done. Also saved to Drive:", DRIVE_CKPT)
        break
else:
    print("❌ No checkpoint found. Run training steps first.")